# Import library

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# Download and create dataloader

In [ ]:
# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# define a basic transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5, 0.5), (0.5,0.5, 0.5))
])

# download dataset CIFAR-10
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

dataloader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)

# Define GAN hyperparameters

In [ ]:
# hyperparameters
latent_dim = 100
num_classes = 10
embedding_dim = 100
lr = 2e-3
beta1 = 0.5
beta2 = 0.999
num_epochs = 10

# Define generator

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim, num_classes, embedding_dim):
        super(Generator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, embedding_dim)
        self.model = nn.Sequential(
            nn.Linear(latent_dim + embedding_dim, 128*8*8),
            nn.ReLU(),
            nn.Unflatten(1, (128, 8, 8)),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128, momentum=0.78),
            nn.ReLU(),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64, momentum=0.78),
            nn.ReLU(),
            nn.Conv2d(64, 3, kernel_size=3, padding=1),
            nn.Tanh()
        )

    def forward(self, z, labels):
        label_input = self.label_emb(labels)
        gen_input = torch.cat((z, label_input), dim=1)
        img = self.model(gen_input)
        return img
        

# Build discriminator

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, num_classes, img_size=32):
        super(Discriminator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, img_size*img_size)

        self.model = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=3, stride=2, padding=1), # 3 channels color + 1 label y
            nn.LeakyReLU(0.2),
            nn.Dropout(0.25),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ZeroPad2d((0, 1, 0, 1)),
            nn.BatchNorm2d(64, momentum=0.82),
            nn.LeakyReLU(0.25),
            nn.Dropout(0.25),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128, momentum=0.82),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.25),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256, momentum=0.8),
            nn.LeakyReLU(0.25),
            nn.Dropout(0.25),
            nn.Conv2d(256, 1, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x, labels):
        label_input = self.label_emb(labels).view(labels.size(0), 1, 32, 32)
        x = torch.cat((x, label_input), dim=1)
        validity = self.model(x)
        return validity

# Initialize GAN component

In [ ]:
generator = Generator(latent_dim, num_classes, embedding_dim).to(device)

discriminator = Discriminator(num_classes).to(device)

# Loss function
adversarial_loss = nn.BCELoss()

# Optimizer
optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, beta2))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, beta2))

# Train

In [ ]:
# training loop
for epoch in range(num_epochs):
    
    for i, (real_images, labels) in enumerate(dataloader):
        batch_size = real_images.size(0)

        real_images = real_images.to(device)
        labels = labels.to(device)

        valid = torch.ones(batch_size, 1, device=device)
        fake = torch.zeros(batch_size, 1, device=device)

        # train generator
        optimizer_G.zero_grad()
        z = torch.randn(batch_size, latent_dim, device=device)
        gen_images = generator(z, labels)
        g_loss = adversarial_loss(discriminator(gen_images, labels), valid)
        g_loss.backward()
        optimizer_G.step()

        # train discriminator
        optimizer_D.zero_grad()
        real_out = discriminator(real_images, labels)
        real_loss = adversarial_loss(real_out, torch.ones_like(real_out))

        fake_out = discriminator(gen_images.detach(), labels)
        fake_loss = adversarial_loss(fake_out, torch.zeros_like(fake_out))
        d_loss = (real_loss + fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()


        if (i + 1) % 100 == 0:
            print(
                f"Epoch [{epoch+1}/{num_epochs}] Batch {i+1}/{len(dataloader)} "
                f"D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f}"
            )
    
    # visualize generation
    if (epoch + 1) % 10 == 0:
        generator.eval()
        with torch.no_grad():
            z = torch.randn(16, latent_dim, device=device)
            label_sample = torch.tensor([i % 10 for i in range(16)], device=device)
            gen_imgs = generator(z, label_sample).cpu()
            grid = torchvision.utils.make_grid(gen_imgs, nrow=4, normalize=True)
            plt.imshow(np.transpose(grid, (1, 2, 0)))
            plt.axis("off")
            plt.title("Epoch {}".format(epoch+1))
            plt.show()
        generator.train()